# FindMine detector run 1 (Kaggle)

1. **Add data** → search `coco-2017-dataset` (by awsaf49) → Add. Then **Add data → Upload** `findmine-labels.zip` as a new dataset named `findmine-labels`.
2. **Settings:** Accelerator = **GPU T4 x1** (or P100), Internet = **On**, Persistence = Files only.
3. **Save Version → Save & Run All (Commit)**. It runs in the background (≈4–6 h); you can close the browser.
4. When it finishes: open the version → **Output** → download `out/findmine12.onnx` and `out/findmine12.json`, and put both in `models/`.

In [ ]:
!pip install -q ultralytics==8.4.182 onnx==1.23.2 onnxslim==0.1.98
import glob, os, shutil
from pathlib import Path

W = Path("/kaggle/working/findmine")
labels = Path(glob.glob("/kaggle/input/**/findmine.yaml", recursive=True)[0]).parent
coco = next((Path(p).parent.parent for p in ["/kaggle/input/coco-2017-dataset/coco2017/train2017/000000000009.jpg"] if os.path.exists(p)), None)     or Path(glob.glob("/kaggle/input/**/train2017/000000000009.jpg", recursive=True)[0]).parent.parent
print("labels:", labels, "| coco:", coco)

shutil.copytree(labels, W, dirs_exist_ok=True)   # labels must be writable (Ultralytics writes .cache files)
(W / "images").mkdir(exist_ok=True)
for split in ("train2017", "val2017"):
    if not (W / "images" / split).exists():
        os.symlink(coco / split, W / "images" / split)
for lst in ("train.txt", "val.txt"):
    paths = (W / lst).read_text().split()
    missing = [p for p in paths if not (W / p).exists()]
    print(lst, len(paths), "images,", len(missing), "missing")
    assert not missing, missing[:5]

In [ ]:
%cd /kaggle/working
!python {W}/train_detector.py --data {W}/findmine.yaml --epochs 70 --batch -1

In [ ]:
import json
from ultralytics import YOLO
best = YOLO("/kaggle/working/runs/detect/findmine_run1/weights/best.pt")
onnx = Path(best.export(format="onnx", opset=12, imgsz=640, simplify=True, dynamic=False))
out = Path("/kaggle/working/out"); out.mkdir(exist_ok=True)
shutil.copy(onnx, out / "findmine12.onnx")
(out / "findmine12.json").write_text(json.dumps([best.names[i] for i in range(len(best.names))]))
shutil.copy("/kaggle/working/runs/detect/findmine_run1/results.csv", out / "results.csv")
print(open(out / "findmine12.json").read())